# 按最终论文顺序复现：数据 → 功能分析 → CLEAN → SepFun

对照 **Thesis_final.pdf（June 2024，77 页）**。本 notebook 按论文第 3、4 章排列；原始文件只读，新结果写入独立目录。数值重算和图形布局的复现程度分别记录。**没有完整数据的图表保留编号并明确列出缺口，不用历史截图代替运行。** 最后一部分才是修改口径和改进分析。

使用 **Thesis reproduction (Python 3.11)** 内核，然后 Restart and Run All。第三方引用示意图（第 2 章、图 3.8–3.19）不是本研究的重算结果；图 3.1–3.7 是实证数据图，纳入复现。本文保留旧代码的 EC 多标签展开口径；绘图软件版本、排序或历史数据快照不确定时明确注明，不宣称逐像素一致。

In [ ]:
import sys, os, json, csv, re, hashlib, datetime, subprocess, shutil
from pathlib import Path
from collections import Counter,defaultdict
if sys.version_info < (3,11):
    raise RuntimeError('请切换到 Thesis reproduction (Python 3.11) 内核再 Run All。当前：'+sys.executable)
import numpy as np,pandas as pd,h5py,matplotlib.pyplot as plt,seaborn as sns
from IPython.display import display
from scipy.stats import hypergeom
ROOT=Path(os.environ.get('THESIS_ROOT',Path.cwd())).resolve()
assert (ROOT/'01_code/Code').is_dir(),'请设置 THESIS_ROOT 为整理后的 Thesis 根目录'
DATA=ROOT/'02_data/data';CLEAN=ROOT/'03_models_and_tools/tttianhao-CLEAN-0cf2cac'
RUN=ROOT/'04_results'/('paper_order_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
RUN.mkdir(parents=True,exist_ok=False);FIG=RUN/'figures';FIG.mkdir()
status=[];plt.rcParams.update({'font.size':10,'axes.grid':False})
def availability(p):
    p=Path(p)
    for q in reversed([p,*p.parents]):
        try:
            if getattr(q.lstat(),'st_flags',0)&0x40000000:
                try:label=q.relative_to(ROOT)
                except ValueError:label=q
                return 'cloud_only: '+str(label)
        except FileNotFoundError:return 'missing: '+str(q)
    return 'local'
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def record_blocked(items,reason,paths):
    checks=[{'path':p,'availability':availability(ROOT/p)} for p in paths]
    status.append(dict(item=items,status='blocked',note=reason,inputs=checks));display(pd.DataFrame(checks))
def savefig(item,fig,note='Recomputed; original panel types restored; not pixel-identical'):
    fig.tight_layout();fig.savefig(FIG/(item.replace(' ','_')+'.png'),dpi=180,bbox_inches='tight');status.append(dict(item=item,status='recomputed_figure',note=note));plt.show();plt.close(fig)
def table(item,d,note):
    d.to_csv(RUN/(item.replace(' ','_')+'.csv'),index=False);display(d);status.append(dict(item=item,status='table',note=note))
print('本次输出:',RUN)
clusters=defaultdict(list)
with h5py.File(DATA/'Fun_fold.h5','r') as f:
    for p in f:
        v=f[p][()];clusters[v.decode() if isinstance(v,bytes) else str(v)].append(p)
(RUN/'environment.json').write_text(json.dumps({'python':sys.version,'pandas':pd.__version__,'numpy':np.__version__,'seaborn':sns.__version__},indent=2))


## 3.1 数据集：图 3.1–3.7

总体使用 split100（227,362 条）；最终模型训练另用 0.5 split。测试使用 new（392 条）。图 3.1/3.6 按原文采用 10 AA 分箱；旧探索代码曾用 bins=100，二者不保证完全相同。顶层 EC 分布按原代码每个 EC 标签展开，多标签蛋白可能重复计数。

In [ ]:
train=pd.read_csv(CLEAN/'data/split100.csv',sep='\t',dtype=str);test=pd.read_csv(CLEAN/'data/new.csv',sep='\t',dtype=str)
all_labels=dict(zip(train.Entry,train['EC number'].str.split(';')))
EC_NAMES=['Oxidoreductases','Transferases','Hydrolases','Lyases','Isomerases','Ligases','Translocases']
def dataset_figures(frame,f_length,f_ec):
    lengths=frame.Sequence.str.len();groups=defaultdict(list);ecs=defaultdict(set)
    for row in frame.itertuples(index=False,name=None):
        p,e,s=row[:3]
        for ec in e.split(';'):
            top=int(ec[0])-1;groups[top].append(len(s));ecs[top].add(ec)
    fig,axs=plt.subplots(1,2,figsize=(11,4));bins=np.arange(0,max(lengths)+11,10)
    axs[0].hist(lengths,bins=bins,color='skyblue',alpha=.5)
    for i,color in enumerate(['blue','orange','green','red','purple','brown','pink']):axs[1].hist(groups[i],bins=bins,alpha=.5,label=EC_NAMES[i],color=color)
    for ax in axs:ax.set(xlabel='Sequence Length',ylabel='Count',xlim=(0,1050))
    axs[1].legend(fontsize=7);savefig(f_length,fig,'10 AA bins as thesis caption; multi-label expansion as original notebook; plotted 0-1050 AA')
    fig,axs=plt.subplots(1,2,figsize=(12,4));x=np.arange(7)
    axs[0].bar(x,[len(groups[i])/len(ecs[i]) if ecs[i] else np.nan for i in x],color='skyblue',alpha=.5)
    axs[1].bar(x,[len(ecs[i]) for i in x],color='skyblue',alpha=.5)
    for ax in axs:ax.set_xticks(x,EC_NAMES,rotation=20,ha='right');ax.set_ylabel('Count')
    axs[0].set_title('Average count per fourth-level functional category',fontsize=10);axs[1].set_title('Count of fourth-level functional categories',fontsize=10)
    return fig
fig32=dataset_figures(train,'Figure 3.1','Figure 3.2');savefig('Figure 3.2',fig32)


### 图 3.3–3.5：训练总体 GO 概览
原图前三类 GO 概览的计数对应结构簇大小 >10 的注释成员，不是未筛选的全部训练集。此处从原始注释按该范围重算并保存分母；BP phosphorylation = 11,960。成员关系在公共准备单元格读取一次。

In [ ]:
GO_COLS={'BP':'Gene Ontology (biological process)','MF':'Gene Ontology (molecular function)','CC':'Gene Ontology (cellular component)'}
go_train={a:{} for a in GO_COLS};go_names={};train_ids={p for members in clusters.values() if len(members)>10 for p in members}
for chunk in pd.read_csv(DATA/'uniprotkb_AND_reviewed_true_2023_07_03.tsv',sep='\t',usecols=['Entry',*GO_COLS.values()],dtype=str,chunksize=30000):
    chunk=chunk[chunk.Entry.isin(train_ids)]
    for a,col in GO_COLS.items():
        for p,s in zip(chunk.Entry,chunk[col]):
            if pd.notna(s):
                terms=set()
                for term in s.split(';'):
                    m=re.search(r'(.*?)\s*\[(GO:\d{7})\]',term)
                    if m:terms.add(m[2]);go_names[m[2]]=m[1].strip()
                if terms:go_train[a][p]=terms
for a,num in [('BP',3),('MF',4),('CC',5)]:
    counts=Counter(t for terms in go_train[a].values() for t in terms);d=pd.DataFrame(counts.most_common(10),columns=['GO','count']);d['term']=d.GO.map(go_names);d['annotated_denominator']=len(go_train[a]);d['percent']=100*d['count']/d.annotated_denominator;d.to_csv(RUN/f'Figure_3.{num}_data.csv',index=False)
    fig,ax=plt.subplots(figsize=(9,4));ax.barh(d.term+' ['+d.GO+']',d['count'],color='skyblue');ax.invert_yaxis();ax.set(xlabel='Count',ylabel='GO Term',title='Top Enriched GO Terms');savefig(f'Figure 3.{num}',fig,'Original count bars; cluster size >10 annotation universe, denominator saved; typography not pixel-identical')


In [ ]:
fig37=dataset_figures(test,'Figure 3.6','Figure 3.7');savefig('Figure 3.7',fig37)

## 3.2–3.5 方法与模型设置

Foldseek 同源性是论文采用的操作定义。CLEAN：EC 监督 triplet 投影、EC 中心距离和 maximum separation 推断。SepFun：EC/Fold 双编码器，min/null/max 三种约束。以下是原文配置，不代表已重跑训练。

In [ ]:
table('Table 3.1',pd.DataFrame([{'Epochs':3500,'Learning rate':5e-4,'Adaptive rate':100,'Hidden dim':512,'Out dim':128}]),'Reported thesis configuration, not training rerun')

## 4.1 数据处理：图 4.1–4.6，表 4.1



**待复现：Figure 4.1–4.5**

缺少原始结构获取清单、分阶段聚类表和严格对应的 pLDDT 快照；不根据目前文件数补造原图。

所需数据：`02_data/data/all`, `02_data/data/alignment_db`

In [ ]:
record_blocked('Figure 4.1–4.5','缺少原始结构获取清单、分阶段聚类表和严格对应的 pLDDT 快照；不根据目前文件数补造原图。',['02_data/data/all', '02_data/data/alignment_db'])

In [ ]:
rows=[]
for k,members in clusters.items():
    labels=[all_labels[p] for p in members if p in all_labels];complete=len(labels)==len(members)
    rows.append(dict(cluster=k,size=len(members),complete_labels=complete,list_consistent=complete and all(e==labels[0] for e in labels),set_consistent=complete and all(set(e)==set(labels[0]) for e in labels)))
cluster_summary=pd.DataFrame(rows);cluster_summary.to_csv(RUN/'cluster_ec_consistency.csv',index=False)
table('Table 4.1',pd.DataFrame({'stage':['Input','Sequence clusters','Representatives','pLDDT filtered','Structure clusters','Expanded members'],'reported':[227362,34280,31685,22980,3291,194521],'rerun':[len(train),None,None,None,len(clusters),sum(map(len,clusters.values()))]}),'Partial: upstream stages not rerun')


In [ ]:
# Streaming universe counts: row counts, not a claim of global unique protein IDs.
universe_counts=Counter();flags=Counter();total_rows=0
if availability(DATA/'Mem_all.txt')=='local':
    with (DATA/'Mem_all.txt').open() as f:
        for line in f:
            fields=line.rstrip('\n').split('\t');total_rows+=1;flags[fields[2]]+=1
            if fields[2] in {'1','2'}:universe_counts[fields[0]]+=1
    links=defaultdict(set);last_local={}
    with (DATA/'Clu_all.txt').open() as f:
        for line in f:
            u,k,*_=line.rstrip('\n').split('\t');links[u].add(k);last_local[u]=k
    mapping_rows=[]
    for u,ks in links.items():
        for k in sorted(ks):mapping_rows.append(dict(universe=u,local=k,local_size=len(clusters.get(k,[])),universe_size=universe_counts.get(u,0)))
    mapping=pd.DataFrame(mapping_rows);mapping.to_csv(RUN/'universe_mapping_recomputed.csv',index=False)
    selected=[u for u,n in universe_counts.items() if 1000<n<2000 and u in last_local and len(clusters.get(last_local[u],[]))>1]
    ureport=dict(total_rows=total_rows,flags=dict(flags),retained_member_rows=sum(universe_counts.values()),universe_clusters=len(universe_counts),mapping_local_ids=mapping.local.nunique(),mapping_global_ids=len(links),multi_global_groups=sum(len(v)>1 for v in links.values()),locals_in_multi_groups=sum(len(v) for v in links.values() if len(v)>1),single_global_groups=sum(len(v)==1 for v in links.values()),unmapped_local_clusters=sorted(set(clusters)-set(mapping.local)),subset_legacy_clusters=len(selected),subset_legacy_local_members=sum(len(clusters[last_local[u]]) for u in selected),subset_universe_member_rows=sum(universe_counts[u] for u in selected),legacy_reverse_mapping_collisions=sum(len(v)-1 for v in links.values()),mean_expansion_per_local=float((mapping.universe_size/mapping.local_size.replace(0,np.nan)).mean()))
    (RUN/'universe_summary.json').write_text(json.dumps(ureport,indent=2))
    cases=mapping[mapping.local.isin(['Q7N2V3','P20921'])];cases.to_csv(RUN/'table4_7_cases.csv',index=False)



In [ ]:
ubins=[(1,2),(2,10),(10,100),(100,500),(500,1000),(1000,2000),(2000,3000),(3000,4000),(4000,6000),(6000,10000),(10000,20000),(20000,30000),(30000,60000),(60000,150000),(150000,600000)]
uv=np.array(list(universe_counts.values()));uc=np.array([((uv>=a)&(uv<b)).sum() for a,b in ubins]);pd.DataFrame({'bin':[str(x) for x in ubins],'count':uc}).to_csv(RUN/'Figure_4.6_bins.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,idx in zip(axs,[range(6),range(6,len(ubins))]):
    ax.bar(range(len(idx)),uc[list(idx)],color='silver');ax.set_xticks(range(len(idx)),[f'{ubins[i][0]}-{ubins[i][1]}' for i in idx],rotation=30,ha='right');ax.set(xlabel='Cluster Size',ylabel='Count')
    for x,i in enumerate(idx):ax.text(x,uc[i],str(uc[i]),ha='center',va='bottom',fontsize=8)
savefig('Figure 4.6',fig,'Two-panel cluster-count distribution; edges recovered from original notebook cell 27')


## 4.2.1 EC 一致性：图 4.7，表 4.2–4.3

主分析使用原代码的 EC 列表一致定义。单成员簇仍保留。集合口径与非单成员敏感性分析移到最后。

In [ ]:
segments=[(1,2),(2,4),(4,10),(10,50),(50,100),(100,300),(300,600),(600,900)]
counts=[]
for a,b in segments:
    d=cluster_summary[(cluster_summary['size']>=a)&(cluster_summary['size']<b)];yes=int(d.list_consistent.sum());counts.append([yes,len(d)-yes])
x=np.arange(len(segments));counts=np.array(counts);pct=100*counts/counts.sum(axis=1,keepdims=True)
fig,axs=plt.subplots(1,2,figsize=(12,4))
for i,(label,color) in enumerate([('EC consistent','gray'),('EC not consistent','salmon')]):
    axs[0].bar(x+(i-.5)*.36,counts[:,i],.36,color=color,alpha=.7,label=label)
    for j,v in enumerate(counts[:,i]):axs[0].text(j+(i-.5)*.36,v,str(v),fontsize=7,ha='center',va='bottom')
    axs[1].bar(x,pct[:,i],bottom=pct[:,0] if i else 0,color=color,alpha=.7,label=label)
for ax in axs:ax.set_xticks(x,[f'{a}-{b}' for a,b in segments],rotation=20);ax.set_xlabel('Cluster Size');ax.legend(fontsize=7)
axs[0].set_ylabel('Count');axs[1].set_ylabel('Percentage');savefig('Figure 4.7',fig)
print('原图分箱未覆盖 size >=900 的簇数:',int(cluster_summary['size'].ge(900).sum()))
table('Table 4.2',pd.DataFrame({'Sequence':[True,True,False,False],'Structure':[True,False,True,False],'reported_counts':[2283,1,121,886],'reported_singletons':[679,0,0,0]}),'Reported 2x2 values; only aggregate 2283/1008 and 679 independently recovered; upstream assignments missing')


In [ ]:
# EC levels: preserve the legacy first-annotation definition and compare all-label sets.
level_rows=[]
legacy_bad=cluster_summary.loc[~cluster_summary.list_consistent,'cluster'].tolist()
for level in (1,2,3,4):
    first_count=set_count=0
    for key in legacy_bad:
        labs=[all_labels[x] for x in clusters[key]]
        first=['.'.join(x[0].split('.')[:level]) for x in labs]
        sets=[frozenset('.'.join(e.split('.')[:level]) for e in x) for x in labs]
        first_count+=len(set(first))==1
        set_count+=len(set(sets))==1
    level_rows.append(dict(level=level,denominator=len(legacy_bad),legacy_first_annotation_consistent=first_count,all_annotation_sets_consistent=set_count))
levels=pd.DataFrame(level_rows);levels.to_csv(RUN/'table4_3_ec_levels.csv',index=False)

levels['reported_consistent']=[713,587,522,np.nan]
table('Table 4.3',levels[['level','denominator','legacy_first_annotation_consistent','reported_consistent']],'Discrepancy: neither first-label nor all-label-prefix reconstruction matches thesis counts')

## 4.2.2 GO 富集：表 4.4，图 4.8–4.15，表 4.5

从原始 UniProt 注释与保存的簇成员关系重算。簇大小 >10，aspect 有注释蛋白作背景，单侧超几何检验和簇内 BH；与原 RDS 对照仅用于验证，不代替计算。

In [ ]:
# Reconstruct GO annotations directly from the dated UniProt TSV, not possibly stale HDF5 exports.
columns={'BP':'Gene Ontology (biological process)','MF':'Gene Ontology (molecular function)','CC':'Gene Ontology (cellular component)'}
eligible={k:v for k,v in clusters.items() if len(v)>10};eligible_ids={p for v in eligible.values() for p in v}
go={a:{} for a in columns}
raw_go=DATA/'uniprotkb_AND_reviewed_true_2023_07_03.tsv'
for chunk in pd.read_csv(raw_go,sep='\t',usecols=['Entry',*columns.values()],chunksize=20000,dtype=str):
    chunk=chunk[chunk.Entry.isin(eligible_ids)]
    for a,col in columns.items():
        for p,terms in zip(chunk.Entry,chunk[col]):
            if pd.notna(terms):
                ids=frozenset(re.findall(r'GO:\d{7}',terms))
                if ids:go[a][p]=ids

def bh_adjust(p):
    p=np.asarray(p,float);order=np.argsort(p);ranked=p[order]*len(p)/np.arange(1,len(p)+1)
    out=np.empty(len(p));out[order]=np.minimum(1,np.minimum.accumulate(ranked[::-1])[::-1]);return out

go_stats=[];go_cluster_rows=[];enrichments=[]
for a,annotations in go.items():
    bg=Counter(t for terms in annotations.values() for t in terms);N=len(annotations);nc=0
    for k,members in eligible.items():
        labeled=[p for p in members if p in annotations]
        if not labeled:continue
        nc+=1;ct=Counter(t for p in labeled for t in annotations[p]);terms=sorted(ct)
        # Exact one-sided over-representation; BH within each cluster over observed terms.
        pv=hypergeom.sf(np.array([ct[t] for t in terms])-1,N,np.array([bg[t] for t in terms]),len(labeled));q=bh_adjust(pv)
        for t,pval,padj in zip(terms,pv,q):enrichments.append(dict(aspect=a,cluster=k,term=t,count=ct[t],sample=len(labeled),bg_count=bg[t],background=N,pvalue=pval,p_adjust=padj))
        go_cluster_rows.append(dict(aspect=a,cluster=k,size=len(members),annotated_members=len(labeled),terms=len(terms),significant_fraction=float((q<.05).mean()),EC_consistent=bool(cluster_summary.set_index('cluster').loc[k,'list_consistent'])))
    go_stats.append(dict(aspect=a,terms=len(bg),annotated_members=N,clusters=nc))
go_stats=pd.DataFrame(go_stats);go_stats.to_csv(RUN/'table4_4_go_recomputed.csv',index=False);display(go_stats)
go_clusters=pd.DataFrame(go_cluster_rows);go_clusters.to_csv(RUN/'go_cluster_recomputed.csv',index=False)
go_enrichment=pd.DataFrame(enrichments);go_enrichment.to_csv(RUN/'go_enrichment_recomputed.csv',index=False)
go_enrichment[(go_enrichment.cluster=='Q7N2V3')&(go_enrichment.aspect=='BP')].to_csv(RUN/'table4_5_go_case_recomputed.csv',index=False)
display(go_clusters.groupby('aspect').agg(mean_terms=('terms','mean'),mean_significant_fraction=('significant_fraction','mean'),all_significant_fraction=('significant_fraction',lambda x:(x==1).mean())))
# Recover stored R results for numerical verification only; these never replace recomputed values.
if shutil.which('Rscript'):
    rcode='''args<-commandArgs(TRUE); for(a in c("BP","MF","CC")){x<-readRDS(file.path(args[1],paste0(a,"_all_data.rds"))); z<-do.call(rbind,lapply(names(x),function(k){d<-x[[k]]; d$cluster<-k; d})); z$aspect<-a; write.csv(z,file.path(args[2],paste0("stored_GO_",a,".csv")),row.names=FALSE)}'''
    subprocess.run(['Rscript','-e',rcode,str(DATA/'GO_analysis'),str(RUN)],check=True,timeout=90,capture_output=True,text=True)
    old=pd.concat([pd.read_csv(RUN/f'stored_GO_{a}.csv') for a in columns])
    joined=go_enrichment.merge(old[['aspect','cluster','ID','pvalue','p.adjust']],left_on=['aspect','cluster','term'],right_on=['aspect','cluster','ID'],how='outer',suffixes=('_rerun','_stored'),indicator=True)
    joined.to_csv(RUN/'go_stored_vs_recomputed.csv',index=False)
    go_check=dict(recomputed_rows=len(go_enrichment),stored_rows=len(old),match_status=joined['_merge'].value_counts().to_dict(),pvalues_close=int(np.isclose(joined.pvalue_rerun,joined.pvalue_stored,rtol=1e-7,atol=1e-300).sum()),adjusted_close=int(np.isclose(joined.p_adjust,joined['p.adjust'],rtol=1e-7,atol=1e-300).sum()))
    (RUN/'go_verification.json').write_text(json.dumps(go_check,indent=2));display(go_check)

table('Table 4.4',go_stats,'Recomputed from raw annotation TSV')

In [ ]:
def violin(ax,arrays,labels,ylim=None):
    sns.violinplot(data=[np.asarray(v) for v in arrays],palette=sns.color_palette('muted',len(arrays)),orient='v',ax=ax,inner='box',cut=2)
    ax.set_xticks(range(len(labels)),labels)
    if ylim:ax.set_ylim(*ylim)
    ax.grid(False)
fig,axs=plt.subplots(1,2,figsize=(10,4));arrays=[go_clusters.loc[go_clusters.aspect==a,'terms'].to_numpy() for a in ['BP','MF','CC']]
for ax in axs:violin(ax,arrays,['BP','MF','CC']);ax.set_ylabel('GO term count')
axs[1].set_ylim(0,30);savefig('Figure 4.8',fig)
fig,axs=plt.subplots(1,3,figsize=(10,4))
for ax,a in zip(axs,['BP','MF','CC']):
    d=go_clusters[go_clusters.aspect==a];violin(ax,[d.loc[d.EC_consistent,'terms'],d.loc[~d.EC_consistent,'terms']],['EC consistent','EC not consistent'],(0,30));ax.set_title(a);ax.tick_params(axis='x',rotation=15)
axs[0].set_ylabel('GO term count');savefig('Figure 4.9',fig)
fig,ax=plt.subplots(figsize=(6,4));violin(ax,[go_clusters.loc[go_clusters.aspect==a,'significant_fraction'] for a in ['BP','MF','CC']],['BP','MF','CC'],(0,1.02));ax.set_ylabel('Proportion below 0.05');savefig('Figure 4.10',fig)
fig,axs=plt.subplots(1,3,figsize=(10,4))
for ax,a in zip(axs,['BP','MF','CC']):
    d=go_clusters[go_clusters.aspect==a];violin(ax,[d.loc[d.EC_consistent,'significant_fraction'],d.loc[~d.EC_consistent,'significant_fraction']],['EC consistent','EC not consistent'],(0,1.02));ax.set_title(a);ax.tick_params(axis='x',rotation=15)
axs[0].set_ylabel('Proportion below 0.05');savefig('Figure 4.11',fig)


In [ ]:
def go_case(key,bar_num,heat_num):
    e=go_enrichment[(go_enrichment.cluster==key)&(go_enrichment.aspect=='BP')].sort_values('p_adjust').head(10).copy();e['description']=e.term.map(go_names).fillna(e.term)
    fig,ax=plt.subplots(figsize=(9,4));norm=plt.Normalize(e.p_adjust.min(),max(e.p_adjust.max(),1e-300));cmap=plt.get_cmap('RdYlBu')
    ax.barh(e.description,e['count'],color=cmap(norm(e.p_adjust)));ax.invert_yaxis();ax.set_xlabel('Count');fig.colorbar(plt.cm.ScalarMappable(norm=norm,cmap=cmap),ax=ax,label='p.adjust');savefig(f'Figure 4.{bar_num}',fig,'Original count bar chart with adjusted-p colour; term order by adjusted p, historical tie order not guaranteed')
    if key=='Q7N2V3':
        t=e.copy();t['GeneRatio']=t['count'].astype(str)+'/'+t['sample'].astype(str);t['BgRatio']=t.bg_count.astype(str)+'/'+t.background.astype(str);table('Table 4.5',t[['term','description','GeneRatio','BgRatio']],'Recomputed ratios')
    members=sorted(clusters[key]);terms=list(e.term);mat=pd.DataFrame([[int(t in go['BP'].get(p,set())) for p in members] for t in terms],index=terms,columns=members);mat.to_csv(RUN/f'Figure_4.{heat_num}_matrix.csv')
    fig,ax=plt.subplots(figsize=(12,5));sns.heatmap(mat,cmap=['white','#303030'],vmin=0,vmax=1,cbar=False,linewidths=.1,linecolor='gray',ax=ax);ax.set_yticklabels([go_names.get(t,t) for t in terms],rotation=0,fontsize=8);ax.set_xticklabels(members,rotation=60,ha='right',fontsize=5);ax.set(xlabel='',ylabel='');savefig(f'Figure 4.{heat_num}',fig,'Raw annotation heatmap; protein IDs explicit, alphabetic order; exact original ordering unavailable')
    return e
caseQ=go_case('Q7N2V3',12,13)
caseP=go_case('P20921',14,15)


## 4.2.3 功能表征：图 4.16–4.18

原图需 Mixed/EC/MFO 三个 aspect，以及相同蛋白排序下的 Sequence/Structure/EC 参照矩阵。不能用单张 Mixed 热图替代六面板原图。

**待复现：Figure 4.16–4.18**

Mixed 本地可用，但 EC/MFO embeddings、原始序列聚类参照及共同排序不足以重建完整原图；Mixed 部分检查放在附录。

所需数据：`02_data/data/Embedding`, `02_data/data/all`

In [ ]:
record_blocked('Figure 4.16–4.18','Mixed 本地可用，但 EC/MFO embeddings、原始序列聚类参照及共同排序不足以重建完整原图；Mixed 部分检查放在附录。',['02_data/data/Embedding', '02_data/data/all'])

## 4.3 未知蛋白宇宙：图 4.19–4.23，表 4.6–4.7



图 4.19–4.20 沿用旧代码反向映射取最后一个本地簇的规则，类别是本地 EC 一致性的继承，不是未知蛋白的实测 EC。原数据快照和当前映射有差异，不强行匹配原图数字。

In [ ]:
# Reproduce the historical reverse-map convention explicitly, not corrected labels.
consistent_keys=set(cluster_summary.loc[cluster_summary.list_consistent,'cluster'])
universe_rows=pd.DataFrame([{'universe':u,'size':n,'assigned_local':last_local[u],'EC_consistent':last_local[u] in consistent_keys} for u,n in universe_counts.items() if u in last_local])
universe_rows.to_csv(RUN/'universe_legacy_categories.csv',index=False)
left_bins=[(1,2),(2,10),(10,100),(100,500),(500,1000),(1000,2000)]
right_bins=[(2000,3000),(3000,4000),(4000,6000),(6000,10000),(10000,20000),(20000,30000),(30000,60000),(60000,150000),(150000,600000)]
fig,axs=plt.subplots(1,2,figsize=(13,4))
for ax,bins in zip(axs,[left_bins,right_bins]):
    for j,(flag,label,color) in enumerate([(True,'EC consistent','gray'),(False,'EC not consistent','salmon')]):
        counts=[int(((universe_rows['size']>=a)&(universe_rows['size']<b)&(universe_rows.EC_consistent==flag)).sum()) for a,b in bins]
        x=np.arange(len(bins));ax.bar(x+(j-.5)*.36,counts,.36,color=color,alpha=.6,label=label)
    ax.set_xticks(range(len(bins)),[f'{a}-{b}' for a,b in bins],rotation=30,ha='right');ax.set(xlabel='Cluster Size',ylabel='Count');ax.legend(fontsize=7)
savefig('Figure 4.19',fig,'Legacy last-local mapping; inherited local EC category, NOT measured universe EC; current snapshot may differ from PDF')
bins=[(1,2),(2,10),(10,2000),(2000,3000),(3000,6000),(6000,60000),(60000,150000),(150000,200000),(200000,600000)]
percent=[]
for a,b in bins:
    d=universe_rows[(universe_rows['size']>=a)&(universe_rows['size']<b)];percent.append(100*d.EC_consistent.mean() if len(d) else np.nan)
fig,ax=plt.subplots(figsize=(11,4));ax.bar(range(len(bins)),percent,color='gray',alpha=.7,label='EC consistent');ax.bar(range(len(bins)),100-np.array(percent),bottom=percent,color='salmon',alpha=.7,label='EC not consistent');ax.set_xticks(range(len(bins)),[f'{a}-{b}' for a,b in bins],rotation=15);ax.set(xlabel='Cluster Size',ylabel='Percentage');ax.legend();savefig('Figure 4.20',fig,'Original notebook bin edges and legacy assignment; empty bins remain NaN')
print('类别由本地簇继承。映射覆盖及正确性解释放在附录；这里仅复现旧计算规则。')


In [ ]:
local_sizes=[len(clusters[last_local[u]]) for u in selected];universe_sizes=[universe_counts[u] for u in selected]
table('Table 4.6',pd.DataFrame([{'set':name,'Clusters':len(v),'Members':sum(v),'Cluster Size Range':f'{min(v)}-{max(v)}','Cluster Size Average':np.mean(v)} for name,v in [('Local',local_sizes),('Universe',universe_sizes)]]),'Recomputed with legacy last-local reverse mapping; collision issue discussed only in appendix')
fig,ax=plt.subplots(figsize=(6,4));violin(ax,[local_sizes,universe_sizes],['Local','Universe']);ax.set_ylabel('Cluster Size');savefig('Figure 4.21',fig)
table('Table 4.7',cases,'Recomputed case mapping sizes')


**待复现：Figure 4.22–4.23**

缺少未知宇宙相同样本的 ProteinVec 多 aspect 表征，无法重画局部/宇宙对照。

所需数据：`02_data/data/Embedding`, `02_data/data/Universe_1000_2000`

In [ ]:
record_blocked('Figure 4.22–4.23','缺少未知宇宙相同样本的 ProteinVec 多 aspect 表征，无法重画局部/宇宙对照。',['02_data/data/Embedding', '02_data/data/Universe_1000_2000'])

## 4.4 CLEAN 验证：表 4.8–4.11，图 4.24–4.25

评估层级：全部 → 未映射 / 映射 → 映射组中的 EC 一致 / 不一致。最终基准是 ProtT5、AugProtT5、ProstT5、AugProstT5；不使用早期 split10 四模型代替。

**待复现：Table 4.8–4.10**

需恢复精确测试映射、逐样本 Foldseek 预测与增强训练清单。当前不重新训练，也不补造指标。

所需数据：`02_data/data/all`, `03_models_and_tools/tttianhao-CLEAN-0cf2cac/results`

In [ ]:
record_blocked('Table 4.8–4.10','需恢复精确测试映射、逐样本 Foldseek 预测与增强训练清单。当前不重新训练，也不补造指标。',['02_data/data/all', '03_models_and_tools/tttianhao-CLEAN-0cf2cac/results'])

In [ ]:
groups=['all','unmapped','mapped','EC_consistent','EC_inconsistent']
values={'ProtT5':([.419,.502,.353,.902,.268],[.474,.483,.450,.911,.345]),'AugProtT5':([.299,.203,.327,.820,.250],[.327,.181,.404,.844,.304]),'ProstT5':([.437,.542,.370,.946,.277],[.492,.509,.475,.933,.371]),'AugProstT5':([.442,.499,.397,.924,.312],[.464,.466,.458,.911,.356])}
benchmark=pd.DataFrame([dict(group=g,model=m,F1=f1[i],accuracy=acc[i],evidence='reported_pdf_not_model_rerun') for m,(f1,acc) in values.items() for i,g in enumerate(groups)])
benchmark.to_csv(RUN/'table4_11_reported.csv',index=False)

table('Table 4.11',benchmark,'Reported PDF values only; NOT rerun predictions')

**待复现：Figure 4.24–4.25**

缺少最终四模型精确 checkpoint/输入对应和多 aspect 参照，不能生成完整的模型对照热图与小提琴图。

所需数据：`02_data/data/Embedding`, `03_models_and_tools/tttianhao-CLEAN-0cf2cac/data/prost5`, `03_models_and_tools/tttianhao-CLEAN-0cf2cac/results`

In [ ]:
record_blocked('Figure 4.24–4.25','缺少最终四模型精确 checkpoint/输入对应和多 aspect 参照，不能生成完整的模型对照热图与小提琴图。',['02_data/data/Embedding', '03_models_and_tools/tttianhao-CLEAN-0cf2cac/data/prost5', '03_models_and_tools/tttianhao-CLEAN-0cf2cac/results'])

## 4.5 SepFun：图 4.26–4.27，表 4.12

双编码器分别以 EC 与 fold ID 为监督，min/null/max 三种约束。数据规模核验不是六模型复现。

In [ ]:
# SepFun training-label evidence (not a model rerun).
with h5py.File(DATA/'sepFun.h5','r') as f:
    sepids=list(f);folds={str(f[k][()]) for k in f}
sep_counts=dict(proteins=len(sepids),fold_ids=len(folds),EC_terms=len({e for p in sepids for e in all_labels.get(p,[])}),missing_EC_ids=sum(p not in all_labels for p in sepids))
(RUN/'sepfun_training_counts.json').write_text(json.dumps(sep_counts,indent=2));display(sep_counts)



**待复现：Figure 4.26–4.27; Table 4.12**

六个编码器的 checkpoint 身份、训练配置、输入 embedding 与逐样本预测不齐全。

所需数据：`03_models_and_tools/sep_clr-master/Protein_Model`, `01_code/Code/src`, `02_data/data/Embedding`

In [ ]:
record_blocked('Figure 4.26–4.27; Table 4.12','六个编码器的 checkpoint 身份、训练配置、输入 embedding 与逐样本预测不齐全。',['03_models_and_tools/sep_clr-master/Protein_Model', '01_code/Code/src', '02_data/data/Embedding'])

In [ ]:
table('Table 4.12 reported',pd.DataFrame({'encoder':['EC min','EC null','EC max','Fold min','Fold null','Fold max'],'F1':[.273,.266,.293,.237,.322,.220],'ACC':[.345,.356,.366,.289,.397,.284]}),'Reported values only; NOT model rerun')

## 附录 A：改进分析与差异解释（不替代原论文复现）

单独报告 singleton 敏感性、多标签集合口径、宇宙映射覆盖、GO 规模分层、增强收益差值和去对角线分析。所有新口径与原图原表分开。

In [ ]:
print('EC 一致率，包含 singleton:',cluster_summary.list_consistent.mean())
print('排除 singleton:',cluster_summary.loc[cluster_summary['size']>1,'list_consistent'].mean())
display(cluster_summary[cluster_summary.list_consistent!=cluster_summary.set_consistent])
print('映射覆盖与筛选口径:');display(ureport)
display(levels)
print('GO 完全显著簇比例:');display(go_clusters.groupby('aspect').significant_fraction.agg(lambda x:(x==1).mean()))


In [ ]:
deltas=[]
for base in ['ProtT5','ProstT5']:
    for i,g in enumerate(groups):deltas.append(dict(model=base,group=g,delta_F1=round(values['Aug'+base][0][i]-values[base][0][i],3),delta_accuracy=round(values['Aug'+base][1][i]-values[base][1][i],3)))
deltas=pd.DataFrame(deltas);deltas.to_csv(RUN/'table4_11_augmentation_deltas.csv',index=False);display(deltas)
fig,axs=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,base in zip(axs,['ProtT5','ProstT5']):
    d=deltas[deltas.model==base];x=np.arange(len(d));ax.barh(x-.17,d.delta_F1,.34,label='F1');ax.barh(x+.17,d.delta_accuracy,.34,label='Accuracy');ax.set_yticks(x,groups);ax.axvline(0,color='black',lw=.7);ax.set_title(base+' augmentation (reported table)');ax.legend()
fig.tight_layout();fig.savefig(RUN/'augmentation_claim_audit.png',dpi=150);plt.show()


In [ ]:
# Additional diagnostics. RUN is the output directory created by the notebook.
# These cells read saved, independently recomputed tables; they do not overwrite sources.
from scipy.stats import hypergeom
import json
GO_RESULTS=pd.read_csv(RUN/'go_cluster_recomputed.csv')
GO_RESULTS['size_bin']=pd.cut(GO_RESULTS['size'],[10,20,50,100,300,1000,float('inf')],right=False)
strat=GO_RESULTS.groupby(['aspect','size_bin','EC_consistent'],observed=True).agg(clusters=('cluster','size'),mean_terms=('terms','mean'),mean_significant_fraction=('significant_fraction','mean')).reset_index()
strat.to_csv(RUN/'go_size_stratified.csv',index=False)
print('按簇规模分层，供检查 EC 一致/不一致差异是否受簇大小影响；不是因果估计。');display(strat)
fig,axs=plt.subplots(2,3,figsize=(13,7))
for j,a in enumerate(['BP','MF','CC']):
    d=GO_RESULTS[GO_RESULTS.aspect==a]
    for flag,label,color in [(True,'EC consistent','gray'),(False,'EC inconsistent','salmon')]:
        v=d[d.EC_consistent==flag]
        axs[0,j].hist(v.terms,bins=30,histtype='step',color=color,label=label)
        axs[1,j].hist(v.significant_fraction,bins=np.linspace(0,1,11),histtype='step',color=color,label=label)
    axs[0,j].set(title=a,xlabel='Unique GO terms',ylabel='Clusters',yscale='log');axs[1,j].set(xlabel='Significant term fraction',ylabel='Clusters');axs[0,j].legend(fontsize=8)
fig.tight_layout();fig.savefig(RUN/'go_consistency_comparison.png',dpi=160);plt.show()

# Reconstruct the two case heatmaps directly from raw UniProt annotations.
case_members={}
with h5py.File(DATA/'Fun_fold.h5','r') as f:
    for p in f:
        v=f[p][()];v=v.decode() if isinstance(v,bytes) else str(v)
        if v in {'Q7N2V3','P20921'}:case_members.setdefault(v,[]).append(p)
ids={p for v in case_members.values() for p in v};case_go={}
for chunk in pd.read_csv(DATA/'uniprotkb_AND_reviewed_true_2023_07_03.tsv',sep='\t',usecols=['Entry','Gene Ontology (biological process)'],chunksize=30000,dtype=str):
    chunk=chunk[chunk.Entry.isin(ids)]
    for p,s in zip(chunk.Entry,chunk['Gene Ontology (biological process)']):case_go[p]=set(re.findall(r'GO:\d{7}',s)) if pd.notna(s) else set()
E=pd.read_csv(RUN/'go_enrichment_recomputed.csv')
for key,members in case_members.items():
    terms=sorted({t for p in members for t in case_go.get(p,set())});members=sorted(members)
    matrix=pd.DataFrame([[int(t in case_go.get(p,set())) for p in members] for t in terms],index=terms,columns=members)
    matrix.to_csv(RUN/f'GO_BP_{key}_presence.csv')
    fig,axs=plt.subplots(1,2,figsize=(12,5));axs[0].imshow(matrix,aspect='auto',interpolation='nearest',vmin=0,vmax=1,cmap='Greys');axs[0].set_yticks(range(len(terms)),terms,fontsize=6);axs[0].set(xlabel='Protein (IDs in CSV)',title=key+' BP annotation')
    e=E[(E.cluster==key)&(E.aspect=='BP')].sort_values('p_adjust').head(10);axs[1].barh(e.term,-np.log10(e.p_adjust.clip(lower=1e-300)));axs[1].invert_yaxis();axs[1].set(xlabel='-log10 BH-adjusted p',title='Top observed terms');fig.tight_layout();fig.savefig(RUN/f'GO_BP_{key}.png',dpi=170);plt.show()

# Available Mixed embeddings: reproduce only these panels; EC/MFO remain unavailable.
cosine_cases=[]
with h5py.File(DATA/'Enzyme_proteins_embed_dic','r') as f,h5py.File(DATA/'Miss_proteins_embed_dic','r') as extra:
    for key,members in case_members.items():
        members=sorted(members);vectors=[];missing=[]
        for p in members:
            source=f if p in f else extra
            if p not in source:missing.append(p);continue
            v=np.asarray(source[p][()]).reshape(-1);vectors.append(v)
        if missing:print('BLOCKED Mixed case',key,missing);continue
        X=np.stack(vectors).astype(float);norm=np.linalg.norm(X,axis=1)
        assert np.all(norm>0) and np.all(np.isfinite(X))
        X/=norm[:,None];S=np.clip(X@X.T,-1,1);off=S[np.triu_indices(len(S),1)]
        pd.DataFrame(S,index=members,columns=members).to_csv(RUN/f'Mixed_cosine_{key}.csv')
        cosine_cases.append(dict(cluster=key,n=len(S),legacy_all_pairs_mean=float(S.mean()),off_diagonal_mean=float(off.mean()),legacy_all_pairs_variance=float(S.var()),off_diagonal_variance=float(off.var())))
        fig,ax=plt.subplots(figsize=(5,4));im=ax.imshow(S,vmin=-1,vmax=1,cmap='viridis');fig.colorbar(im,ax=ax);ax.set(title=key+' Mixed embedding',xlabel='Protein index',ylabel='Protein index');fig.tight_layout();fig.savefig(RUN/f'Mixed_cosine_{key}.png',dpi=160);plt.show()
pd.DataFrame(cosine_cases).to_csv(RUN/'mixed_case_statistics.csv',index=False);display(pd.DataFrame(cosine_cases))
# Re-evaluate the one extreme-tail numerical discrepancy without changing the original R evidence.
z=pd.read_csv(RUN/'go_stored_vs_recomputed.csv');bad=z[~np.isclose(z.p_adjust,z['p.adjust'],rtol=1e-7,atol=1e-300)].copy()
if len(bad):
    bad['log_p_stable']=hypergeom.logsf(bad['count'].to_numpy()-1,bad.background.to_numpy(),bad.bg_count.to_numpy(),bad['sample'].to_numpy())
    bad['p_stable']=np.exp(bad.log_p_stable);bad.to_csv(RUN/'go_extreme_tail_diagnostic.csv',index=False);display(bad[['aspect','cluster','term','pvalue_rerun','pvalue_stored','p_stable']])
print('Supplement complete:',RUN)


## 附录 B：运行状态、缺失输入和可追溯性



In [ ]:
report=pd.DataFrame([{k:v for k,v in row.items() if k!='inputs'} for row in status]);display(report);report.to_csv(RUN/'figure_table_status.csv',index=False)
(RUN/'missing_inputs.json').write_text(json.dumps([x for x in status if x['status']=='blocked'],ensure_ascii=False,indent=2))
(RUN/'run_manifest.json').write_text(json.dumps({'output':str(RUN),'source_pdf':str(Path(os.environ.get('THESIS_FINAL_PDF','Thesis_final.pdf'))),'source_pdf_sha256':sha256(Path(os.environ.get('THESIS_FINAL_PDF','Thesis_final.pdf'))),'original_figures_rendered':sum(x['status'] in ['recomputed_figure','partial_figure'] for x in status),'model_training_rerun':False,'final_benchmark_inference_rerun':False},indent=2))
print('运行完成。原编号图:',FIG,'；缺失输入:',RUN/'missing_inputs.json')
